In [2]:
import pandas as pd
import numpy as np
import os

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

In [3]:
DATASET_PATH = "dataset"
CSV_PATH = "labels.csv"

df = pd.read_csv(CSV_PATH)

classes = [
    "acne_inflammatory",
    "acne_noninflammatory",
    "hyperpigmentation",
    "wrinkles"
]

In [4]:
from tensorflow.keras.utils import Sequence

class DataGenerator(Sequence):
    def __init__(self, df, batch_size=32):
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size

    def __len__(self):
        return len(self.df) // self.batch_size

    def __getitem__(self, index):
        batch_df = self.df.iloc[index*self.batch_size:(index+1)*self.batch_size]

        images = []
        labels = []

        for i in range(len(batch_df)):
            img_path = os.path.join(DATASET_PATH, batch_df.iloc[i]['filename'])

            try:
                img = load_img(img_path, target_size=(224, 224))
                img = img_to_array(img) / 255.0

                label = batch_df.iloc[i][classes].values.astype(float)

                images.append(img)
                labels.append(label)
            except:
                print("Error:", img_path)

        return np.array(images), np.array(labels)

In [5]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)
train_gen = DataGenerator(train_df, batch_size=32)
val_gen = DataGenerator(val_df, batch_size=32)

In [7]:
from tensorflow.keras.layers import GlobalAveragePooling2D

In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

#base_model.trainable = False
for layer in base_model.layers[:-30]:
    layer.trainable = False

for layer in base_model.layers[-30:]:
    layer.trainable = True

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    #Dense(128, activation='relu'),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(len(classes), activation='sigmoid')
])

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy', 'AUC', 'Precision', 'Recall']
)

In [10]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10
)

C:\Users\HP\AppData\Roaming\Python\Python311\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 158s 752ms/step - accuracy: 0.4667 - loss: 0.0865 - val_accuracy: 0.4744 - val_loss: 0.0672
Epoch 2/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 154s 758ms/step - accuracy: 0.4946 - loss: 0.0366 - val_accuracy: 0.4906 - val_loss: 0.0458
Epoch 3/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 159s 782ms/step - accuracy: 0.5003 - loss: 0.0249 - val_accuracy: 0.4938 - val_loss: 0.0387
Epoch 4/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 175s 865ms/step - accuracy: 0.5025 - loss: 0.0165 - val_accuracy: 0.5056 - val_loss: 0.0540
Epoch 5/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 147s 725ms/step - accuracy: 0.5058 - loss: 0.0125 - val_accuracy: 0.4938 - val_loss: 0.0404
Epoch 6/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 146s 721ms/step - accuracy: 0.5088 - loss: 0.0079 - val_accuracy: 0.5106 - val_loss: 0.0423
Epoch 7/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 146s 720ms/step - accuracy: 0.5125 - loss: 0.0066 - val_accuracy: 0.4975 - val_loss: 0.0442
Epoch 8/10
203/203 ━━━━━━━━━━━━━━━━━━━━ 146s 718ms/step - accuracy: 0.5122 -

In [11]:
model.save("skin_model.h5")